# Google ADK — Agent Development Kit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dhelamay/CPUT_ICAILY_AI2026_MaganLLM_WORKSHOP/blob/main/SA_LIB_WORKSHOP_Day2/04_google_adk/04_google_adk.ipynb)

**Author:** Dr. Nasser Mooman · Magan AI Inc. · nmooman@gmail.com · written with help from Claude (Anthropic)

> ⚠️ **Teaching code — do not use in production.** Short on purpose: no authentication, input validation, prompt-injection protection, error handling or tests. LLM answers can be wrong. Provided as is, without warranty.

**SA-LIB AI Workshop 2026 — Day 2: AI Agents**

Google's agent framework. Gemini works natively; every other provider goes through LiteLLM. Tools are plain Python functions. Locally you can also try the chat UI with `adk web`.

Same example in every framework: **(1) first agent → (2) tools → (3) memory → (4) fallback → (5) multi-agent → complete agent**.

## 0. Setup
Install the packages (takes ~1 minute in Colab).

In [ ]:
%pip install -q "google-adk>=1.0" "litellm" "python-dotenv" "ddgs"

### API keys
**Bring your own LLM API key** (Groq and Gemini have free tiers). No key is needed only if you already run a local server (Ollama, SGLang or vLLM): set `LLM_PROVIDER=ollama`, `OLLAMA_BASE_URL` and `LLM_MODEL`.

* **Colab:** click the 🔑 *Secrets* icon on the left, add e.g. `GROQ_API_KEY`, and enable notebook access.
* **VS Code / Jupyter / Linux:** put your keys in the `.env` file in the repo root (copy `.env.example`).
* Otherwise you'll be asked to paste a key below.

Choose the model provider with `LLM_PROVIDER`: `groq` · `gemini` · `openai` · `deepseek` · `openrouter` · `ollama`.

In [ ]:
import os, getpass
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))          # VS Code / Linux: read ../.env
try:                                           # Colab: read the Secrets panel
    from google.colab import userdata
    for key in ['GROQ_API_KEY', 'GOOGLE_API_KEY', 'OPENAI_API_KEY', 'DEEPSEEK_API_KEY', 'OPENROUTER_API_KEY', 'LANGSMITH_API_KEY', 'LANGFUSE_PUBLIC_KEY', 'LANGFUSE_SECRET_KEY', 'LANGFUSE_HOST', 'LLM_PROVIDER', 'LLM_MODEL']:
        try:
            os.environ[key] = userdata.get(key)
        except Exception:
            pass
except ImportError:
    pass

os.environ.setdefault("LLM_PROVIDER", "groq")   # <- change the provider here if you like
provider = os.environ["LLM_PROVIDER"]
key_name = {"groq": "GROQ_API_KEY", "gemini": "GOOGLE_API_KEY", "openai": "OPENAI_API_KEY",
            "deepseek": "DEEPSEEK_API_KEY", "openrouter": "OPENROUTER_API_KEY"}.get(provider)
if key_name and not os.getenv(key_name):
    os.environ[key_name] = getpass.getpass(f"{key_name}: ")
print("provider:", provider)

### Shared helper: `workshop_common.py`
Reads the provider settings, and holds the two tools every example uses: `web_search` (free DuckDuckGo) and `word_count`.

In [ ]:
%%writefile workshop_common.py
"""
workshop_common.py — shared helpers for every framework folder in this workshop.

(the SAME file is copied into each folder so every folder works on its own —
 edit shared/workshop_common.py and run `python tools/sync_common.py` to update the copies)

What it gives you:
  get_llm_config()   -> which provider / model / base_url / api key to use (read from .env)
  fallback_order()   -> the list of providers to try, in order, for the fallback examples
  ping()             -> quick "are you alive?" check against a provider
  web_search()       -> free DuckDuckGo search (no key)
  word_count()       -> the tiny "tool you wrote yourself" from the original guide

Every provider below speaks the OpenAI chat-completions protocol, which is why every
framework in this workshop can use every provider: we just change base_url + api_key + model.
"""

from __future__ import annotations

import os
from dataclasses import dataclass

from dotenv import find_dotenv, load_dotenv

# walks up from where you run python, so the .env in the repo root is always found
load_dotenv(find_dotenv(usecwd=True))

PROVIDERS = {
    #  name        base_url                                                     api-key env var       default model
    "groq":       ("https://api.groq.com/openai/v1",                           "GROQ_API_KEY",       "llama-3.3-70b-versatile"),
    "gemini":     ("https://generativelanguage.googleapis.com/v1beta/openai/", "GOOGLE_API_KEY",     "gemini-2.5-flash"),
    "openai":     ("https://api.openai.com/v1",                                "OPENAI_API_KEY",     "gpt-4o-mini"),
    "deepseek":   ("https://api.deepseek.com",                                 "DEEPSEEK_API_KEY",   "deepseek-chat"),
    "openrouter": ("https://openrouter.ai/api/v1",                             "OPENROUTER_API_KEY", "meta-llama/llama-3.3-70b-instruct:free"),
    "ollama":     (os.getenv("OLLAMA_BASE_URL", "http://localhost:11434/v1"),  None,                 "qwen2.5:7b"),
}


@dataclass
class LLMConfig:
    provider: str
    model: str
    base_url: str
    api_key: str

    @property
    def litellm_model(self) -> str:
        """Model name in LiteLLM format (used by CrewAI and Google ADK): 'openai/<model>'
        means 'an OpenAI-compatible endpoint' — combined with base_url it works for every provider."""
        return f"openai/{self.model}"


def get_llm_config(provider: str | None = None) -> LLMConfig:
    """Read LLM_PROVIDER / LLM_MODEL / <PROVIDER>_API_KEY from the environment."""
    provider = (provider or os.getenv("LLM_PROVIDER") or "groq").strip().lower()
    if provider not in PROVIDERS:
        raise ValueError(f"Unknown LLM_PROVIDER '{provider}'. Choose one of: {', '.join(PROVIDERS)}")
    base_url, key_env, default_model = PROVIDERS[provider]
    api_key = os.getenv(key_env) if key_env else "ollama"  # ollama runs locally, no key
    if not api_key:
        raise RuntimeError(f"{key_env} is missing — add it to your .env file (see .env.example).")
    # LLM_MODEL only overrides the model of the MAIN provider, not of the fallbacks
    main_provider = (os.getenv("LLM_PROVIDER") or "groq").strip().lower()
    model = (os.getenv("LLM_MODEL") if provider == main_provider else None) or default_model
    return LLMConfig(provider, model, base_url, api_key)


def fallback_order() -> list[str]:
    """Providers to try, in order. Default: your LLM_PROVIDER first, then LLM_FALLBACKS (groq,gemini)."""
    main = (os.getenv("LLM_PROVIDER") or "groq").strip().lower()
    extra = [p.strip().lower() for p in os.getenv("LLM_FALLBACKS", "groq,gemini").split(",") if p.strip()]
    return list(dict.fromkeys([main, *extra]))  # remove duplicates, keep order


def ping(cfg: LLMConfig) -> None:
    """Send a 1-token request. Raises an exception if the provider is down, rate-limited or the key is wrong."""
    from openai import OpenAI

    OpenAI(base_url=cfg.base_url, api_key=cfg.api_key, timeout=20, max_retries=0).chat.completions.create(
        model=cfg.model, messages=[{"role": "user", "content": "ping"}], max_tokens=1
    )


def web_search(query: str) -> str:
    """Search the web with DuckDuckGo (free, no key) and return the top results as text."""
    try:
        from ddgs import DDGS

        results = DDGS().text(query, max_results=3)
        return "\n".join(f"- {r['title']}: {r['body']} ({r['href']})" for r in results) or "no results"
    except Exception as e:  # the free search is rate-limited: return the error instead of crashing
        return f"web search failed ({e}). try again in a few seconds."


def word_count(text: str) -> int:
    """Count how many words are in a piece of text."""
    return len(text.split())

## 1. `01_first_agent.py`

Google ADK 01 — your first agent
ADK pieces: Agent (model + instruction + tools), Runner (runs the loop), Session (the conversation).
Gemini is used natively; every other provider goes through LiteLLM.

In [ ]:
import asyncio

from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import InMemoryRunner
from google.genai import types

from workshop_common import get_llm_config

cfg = get_llm_config()
# gemini -> native ADK model name; anything else -> LiteLLM pointing at an OpenAI-compatible endpoint
model = cfg.model if cfg.provider == "gemini" else LiteLlm(model=cfg.litellm_model, api_base=cfg.base_url, api_key=cfg.api_key)

agent = Agent(
    name="assistant",
    model=model,
    instruction="You are a helpful assistant. Be concise and accurate.",
)


async def main():
    runner = InMemoryRunner(agent=agent, app_name="workshop")
    session = await runner.session_service.create_session(app_name="workshop", user_id="m0h")
    message = types.Content(role="user", parts=[types.Part(text="explain what an AI agent is in two sentences.")])

    # the runner streams EVENTS (model calls, tool calls, ...). we print the final answer.
    async for event in runner.run_async(user_id="m0h", session_id=session.id, new_message=message):
        if event.is_final_response() and event.content and event.content.parts:
            print(f"[{cfg.provider}:{cfg.model}]", event.content.parts[0].text)


if __name__ == "__main__":
    await main()

## 2. `02_agent_with_tools.py`

Google ADK 02 — giving your agent tools
in ADK a tool is just a python function with type hints and a docstring. no decorator needed!

In [ ]:
import asyncio

from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import InMemoryRunner
from google.genai import types

from workshop_common import get_llm_config, web_search

cfg = get_llm_config()
model = cfg.model if cfg.provider == "gemini" else LiteLlm(model=cfg.litellm_model, api_base=cfg.base_url, api_key=cfg.api_key)


# tool 2 — a function you wrote yourself (tool 1, web_search, comes from workshop_common.py)
def word_count(text: str) -> int:
    """Count how many words are in a piece of text."""
    return len(text.split())


agent = Agent(
    name="assistant",
    model=model,
    instruction="You are a helpful assistant. Use your tools when they help answer accurately.",
    tools=[web_search, word_count],
)


async def main():
    runner = InMemoryRunner(agent=agent, app_name="workshop")
    session = await runner.session_service.create_session(app_name="workshop", user_id="m0h")
    text = "search for the latest langchain version, then tell me how many words your answer is."
    message = types.Content(role="user", parts=[types.Part(text=text)])

    async for event in runner.run_async(user_id="m0h", session_id=session.id, new_message=message):
        for call in event.get_function_calls():            # watch the agent loop
            print(f"  tool call -> {call.name}({call.args})")
        if event.is_final_response() and event.content and event.content.parts:
            print(event.content.parts[0].text)


if __name__ == "__main__":
    await main()

## 3. `03_agent_with_memory.py`

Google ADK 03 — giving your agent memory
the SESSION is the memory. same session id = the agent sees the whole conversation.
(InMemorySessionService forgets on exit; DatabaseSessionService("sqlite:///memory.db") persists.)

In [ ]:
import asyncio

from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import InMemoryRunner
from google.genai import types

from workshop_common import get_llm_config

cfg = get_llm_config()
model = cfg.model if cfg.provider == "gemini" else LiteLlm(model=cfg.litellm_model, api_base=cfg.base_url, api_key=cfg.api_key)
agent = Agent(name="assistant", model=model, instruction="You are a helpful assistant. Be concise and accurate.")
runner = InMemoryRunner(agent=agent, app_name="workshop")


async def ask(session_id: str, text: str) -> str:
    message = types.Content(role="user", parts=[types.Part(text=text)])
    answer = ""
    async for event in runner.run_async(user_id="m0h", session_id=session_id, new_message=message):
        if event.is_final_response() and event.content and event.content.parts:
            answer = event.content.parts[0].text   # keep looping: let the runner finish cleanly
    return answer


async def main():
    session = await runner.session_service.create_session(app_name="workshop", user_id="m0h", session_id="chat-1")
    print(await ask(session.id, "hi! my name is m0h."))
    print(await ask(session.id, "what's my name?"))


if __name__ == "__main__":
    await main()

## 4. `04_agent_with_fallback.py`

Google ADK 04 — the fallback setup

In [ ]:
import asyncio

from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import InMemoryRunner
from google.genai import types

from workshop_common import fallback_order, get_llm_config, ping


def get_model():
    for provider in fallback_order():
        try:
            cfg = get_llm_config(provider)
            ping(cfg)
            print(f"using {provider} ({cfg.model})")
            if cfg.provider == "gemini":
                return cfg.model
            return LiteLlm(model=cfg.litellm_model, api_base=cfg.base_url, api_key=cfg.api_key)
        except Exception as e:
            print(f"{provider} failed ({str(e)[:80]}); trying the next one")
    raise RuntimeError("every provider failed — check your keys in .env")


agent = Agent(name="assistant", model=get_model(), instruction="You are a helpful assistant. Be concise.")


async def main():
    runner = InMemoryRunner(agent=agent, app_name="workshop")
    session = await runner.session_service.create_session(app_name="workshop", user_id="m0h")
    message = types.Content(role="user", parts=[types.Part(text="say hi and tell me which model you are.")])
    async for event in runner.run_async(user_id="m0h", session_id=session.id, new_message=message):
        if event.is_final_response() and event.content and event.content.parts:
            print(event.content.parts[0].text)


if __name__ == "__main__":
    await main()

## 5. `05_multi_agent.py`

Google ADK 05 — multi-agent: a SequentialAgent pipeline
researcher -> writer. each agent saves its answer into the shared session STATE with output_key,
and the next agent reads it with {placeholder} in its instruction.
(ADK also has ParallelAgent, LoopAgent, and LLM-driven delegation via sub_agents=[...])

In [ ]:
import asyncio

from google.adk.agents import Agent, SequentialAgent
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import InMemoryRunner
from google.genai import types

from workshop_common import get_llm_config, web_search, word_count

cfg = get_llm_config()
model = cfg.model if cfg.provider == "gemini" else LiteLlm(model=cfg.litellm_model, api_base=cfg.base_url, api_key=cfg.api_key)

researcher = Agent(
    name="researcher",
    model=model,
    instruction="Use web_search to find current facts about the user's topic. Reply with 3 short bullet facts.",
    tools=[web_search],
    output_key="research",            # -> saved to session.state["research"]
)
writer = Agent(
    name="writer",
    model=model,
    instruction="Write one friendly paragraph under 60 words for workshop attendees, using ONLY these facts:\n"
                "{research}\nCheck the length with word_count before answering.",   # <- reads the state
    tools=[word_count],
    output_key="paragraph",
)
pipeline = SequentialAgent(name="team", sub_agents=[researcher, writer])


async def main():
    runner = InMemoryRunner(agent=pipeline, app_name="workshop")
    session = await runner.session_service.create_session(app_name="workshop", user_id="m0h")
    message = types.Content(role="user", parts=[types.Part(text="the latest langchain version")])
    async for event in runner.run_async(user_id="m0h", session_id=session.id, new_message=message):
        if event.is_final_response() and event.content and event.content.parts and event.content.parts[0].text:
            print(f"\n[{event.author}]\n{event.content.parts[0].text}")


if __name__ == "__main__":
    await main()

## 6. `agent.py`

Google ADK — the complete agent (fallback + tools + memory)

In [ ]:
import asyncio

from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import InMemoryRunner
from google.genai import types

import workshop_common as wc


def get_model():
    for provider in wc.fallback_order():
        try:
            cfg = wc.get_llm_config(provider)
            wc.ping(cfg)
            print(f"using {provider} ({cfg.model})")
            if cfg.provider == "gemini":
                return cfg.model
            return LiteLlm(model=cfg.litellm_model, api_base=cfg.base_url, api_key=cfg.api_key)
        except Exception as e:
            print(f"{provider} failed ({str(e)[:80]}); trying the next one")
    raise RuntimeError("every provider failed — check your keys in .env")


agent = Agent(
    name="assistant",
    model=get_model(),
    instruction="You are a helpful assistant. Use your tools when they help answer accurately, and remember the conversation.",
    tools=[wc.web_search, wc.word_count],
)
runner = InMemoryRunner(agent=agent, app_name="workshop")


async def ask(session_id: str, text: str) -> str:
    message = types.Content(role="user", parts=[types.Part(text=text)])
    answer = ""
    async for event in runner.run_async(user_id="m0h", session_id=session_id, new_message=message):
        if event.is_final_response() and event.content and event.content.parts:
            answer = event.content.parts[0].text   # keep looping: let the runner finish cleanly
    return answer


async def main():
    session = await runner.session_service.create_session(app_name="workshop", user_id="m0h")
    print(await ask(session.id, "my name is m0h. search for the latest langchain version, then tell me how many "
                                "words your answer is."))
    print(await ask(session.id, "what's my name?"))


if __name__ == "__main__":
    await main()

---
*SA-LIB AI Workshop 2026 · based on [build-ai-agents-free](https://github.com/Moh4696/build-ai-agents-free) by m0h (MIT)*